# QDIST v4.0.0 reviewed analytical preflight

This notebook standardizes the already-frozen QDIST v3.1.1 hard-clipping detector under the common G1-G10 and A-J framework. It does not overwrite the v3.1.1 freeze and does not change detector thresholds. The v4.0.0 family can proceed only if later cohort extraction proves exact numerical equivalence to the frozen v3.1.1 measurement.

QDIST measures native-waveform evidence compatible with hard clipping or saturation. It does not estimate total harmonic distortion, soft clipping, limiting, dynamic-range compression, automatic gain control, general codec distortion, or perceptual distortion.

In [ ]:
from pathlib import Path
import json
import subprocess
import sys

def find_project_root():
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        if (candidate / 'pyproject.toml').exists() and (candidate / 'src' / 'paper1_qc').exists():
            return candidate
    raise FileNotFoundError('Open this notebook from the paper_1 repository.')

ROOT = find_project_root()
for source in [ROOT / 'src', ROOT / 'src reviewed']:
    if str(source) not in sys.path:
        sys.path.insert(0, str(source))

from paper1_qc_reviewed.qdist_v400 import (
    MEASUREMENT_VERSION, LEGACY_MEASUREMENT_VERSION, ANALYSIS_FEATURES,
    PREFLIGHT_HOTFIX_REVISION, PREFLIGHT_PANEL_STEMS,
    run_preflight, write_json,
)

OUTPUT_ROOT = ROOT / 'outputs reviewed' / 'nonlinear_distortion' / 'qdist-v4.0.0-candidate'
RUN_PACKAGE_TESTS = True
RUN_CODEC_CHARACTERIZATION = True
RUN_COHORT_EXTRACTION = False
PUBLISH_AND_FREEZE = False
SCIENTIFIC_REVIEW_DECISION = "PENDING"

DECLARED_PREFLIGHT_PANELS = (
    "A_construct_response",
    "B_discriminant_specificity",
    "C_transformation_contract",
)
assert DECLARED_PREFLIGHT_PANELS == PREFLIGHT_PANEL_STEMS

print('Project:', ROOT)
print('Reviewed measurement:', MEASUREMENT_VERSION)
print('Frozen numerical baseline:', LEGACY_MEASUREMENT_VERSION)
print('Analysis features:', ANALYSIS_FEATURES)
print('Preflight hotfix:', PREFLIGHT_HOTFIX_REVISION)
print('Declared preflight panels:', DECLARED_PREFLIGHT_PANELS)
print('Output root:', OUTPUT_ROOT)

In [ ]:
package_tests_passed = False
package_test_output = ''
if RUN_PACKAGE_TESTS:
    command = [
        str(ROOT / '.venv' / 'Scripts' / 'python.exe'), '-m', 'pytest',
        str(ROOT / 'tests reviewed' / 'test_qdist_v400.py'),
        str(ROOT / 'tests reviewed' / 'test_qdist_v400_notebook.py'), '-q',
    ]
    completed = subprocess.run(command, cwd=ROOT, capture_output=True, text=True)
    package_test_output = completed.stdout + completed.stderr
    print(package_test_output)
    package_tests_passed = completed.returncode == 0
    if not package_tests_passed:
        raise RuntimeError('Reviewed QDIST package tests failed.')
else:
    print('Package tests skipped by control.')

In [ ]:
evidence = run_preflight(OUTPUT_ROOT, run_codecs=RUN_CODEC_CHARACTERIZATION)
checks = evidence['checks']
figure_index = evidence['figure_index']
manifest = evidence['manifest']
checks

In [ ]:
failed = checks.loc[~checks['passed'].astype(bool)]
print('Blocking checks:', f"{int(checks['passed'].sum())}/{len(checks)}")
print('Figure panels:', sorted(figure_index['panel'].tolist()))
if len(failed):
    display(failed)
    raise RuntimeError('QDIST reviewed preflight has failed blocking checks.')
if RUN_COHORT_EXTRACTION:
    raise RuntimeError('The analytical preflight must not run cohort extraction.')

In [ ]:
manifest_path = OUTPUT_ROOT / 'manifests' / 'qdist_v400_preflight_candidate_manifest.json'
manifest = json.loads(manifest_path.read_text(encoding='utf-8'))
manifest['package_tests_passed'] = bool(package_tests_passed)
manifest['candidate_only'] = True
manifest['cohort_extraction_completed'] = False
manifest['freeze_allowed'] = False
manifest['publish_and_freeze'] = False
manifest['scientific_review_decision'] = SCIENTIFIC_REVIEW_DECISION
manifest['family_scalar_constructed'] = False
manifest['standalone_gate_allowed'] = False
write_json(manifest_path, manifest)
manifest

In [ ]:
assert manifest['preflight_hotfix_revision'] == 'legacy-api-compat-r1'
assert manifest['preflight_blocking_checks_pass']
assert manifest['package_tests_passed']
assert manifest['panels_complete'] == ['A', 'B', 'C']
assert manifest['panel_i_status'] == 'APPLICABLE_pending_event_verification'
assert not manifest['cohort_extraction_completed']
assert not manifest['freeze_allowed']
assert not manifest['family_scalar_constructed']
assert not manifest['standalone_gate_allowed']
print('QDIST v4.0.0 REVIEWED PREFLIGHT COMPLETE')
print('Candidate only. Cohort extraction, event verification, G7-G10 decisions, and freezing remain pending.')